In [84]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

Adding the area of the regions to the dataframe.

First, we'll create a dictionary with area codes and areas, then we iterate over a list of area codes from the accidents, create a new array based on them and add it to the dataframe.

In [85]:
accidents = pd.read_csv("koeti_riista_tau_001_en.csv", encoding='iso-8859-1')

regions = pd.read_csv('maakunnat_pinta_alat.csv')

areas = list(regions.iloc[:, 5])
area_codes = list(regions.iloc[:, 2])
areas_dict = dict(zip(area_codes, areas))

accident_areas = list(accidents.iloc[:, 8])

areas_arr = []

for accident_area in accident_areas:
    areas_arr.append(int(areas_dict[accident_area]))

accidents['regionArea'] = areas_arr

Deriving time of day in seconds from eventTime.

First, we'll assign the values from the eventTime column into an array, then we'll iterate over the arrays strings and convert them to seconds, finally we append them to an array and add it to the dataframe.

In [86]:
seconds_arr = []

times = accidents.iloc[:, [1]].values

# '2017-12-10T16:55:00.000+02:00'

for time in times:
    times_of_day = time[0].split('T')[1]
    tz = times_of_day.split('+')[1].split(':')[0][1]
    time_of_day = times_of_day.split('+')[0].split('.')[0].split(':')
    seconds = (int(time_of_day[0]) + int(tz)) * 3600 + int(time_of_day[1]) * 60 + int(time_of_day[2])
    seconds_arr.append(seconds)

accidents['timeOfDay'] = seconds_arr

accidents

,id,eventTime,year,month,x,y,municipality,municipalityName,region,regionName,roadType,roadTypeShort,roadNumber,roadMaintenance,roadMaintenanceName,wildlifeSpecies,wildlifeSpeciesName,regionArea,timeOfDay
0,1,2017-12-10T16:55:00.000+02:00,2017,12,278773.65900,6.980878e+06,743,Seinäjoki,14,Etelä-Pohjanmaa,3,Valtatie,18.0,1.0,Valtio,47507,Metsäkauris,13864807779,68100
1,2,2017-09-03T13:00:00.000+03:00,2017,9,397501.00800,7.503084e+06,261,Kittilä,19,Lappi,4,Kantatie,80.0,1.0,Valtio,47503,Hirvi,94206757384,57600
2,3,2017-09-18T07:00:00.000+03:00,2017,9,418353.84100,7.514105e+06,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384,36000
3,4,2017-12-01T17:00:00.000+02:00,2017,12,488970.82700,6.722504e+06,285,Kotka,8,Kymenlaakso,5,Seututie,357.0,1.0,Valtio,47629,Valkohäntäpeura,4622479273,68400
4,5,2017-09-18T07:00:00.000+03:00,2017,9,418766.58000,7.506931e+06,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384,36000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63937,63938,2021-10-04T09:30:00.000+03:00,2021,10,236501.85584,6.691606e+06,445,Parainen,2,Varsinais-Suomi,5,Seututie,180.0,1.0,Valtio,47507,Metsäkauris,10574665809,45000
63938,63939,2021-09-20T19:51:48.017+03:00,2021,9,515750.42581,6.946907e+06,778,Suonenjoki,11,Pohjois-Savo,3,Valtatie,9.0,1.0,Valtio,47503,Hirvi,17589162923,82308
63939,63940,2021-05-29T06:00:00.000+03:00,2021,5,245406.95714,6.712757e+06,202,Kaarina,2,Varsinais-Suomi,4,Kantatie,40.0,1.0,Valtio,47507,Metsäkauris,10574665809,32400
63940,63941,2021-04-13T14:30:00.000+03:00,2021,4,238690.96803,6.705560e+06,853,Turku,2,Varsinais-Suomi,7,Puuttuu,NaN,99.0,Ei tietoa,47507,Metsäkauris,10574665809,63000


Converting species names to one-hot values

With pandas get_dummies function, we'll first generate a dataframe with one-hot values for every species. Then, for the merge function to work, we add an id column to it and merge with accidents dataframe.

In [87]:
species_oh = pd.get_dummies(accidents['wildlifeSpeciesName'], dtype=int)

species_oh['id'] = range(len(species_oh))

accidents = accidents.merge(species_oh, on='id')

Next we'll drop unnecessary columns and then rows with null values.

In [88]:
print(len(accidents))

print(len(accidents.dropna()))

accidents = accidents.drop(columns=['eventTime', 'municipalityName', 'regionName', 'roadTypeShort', 'roadMaintenanceName', 'wildlifeSpecies', 'wildlifeSpeciesName', 'roadNumber', 'roadMaintenance'])

accidents = accidents.dropna()

print(len(accidents))

63941
54511
63937


In [89]:
accidents

,id,year,month,x,y,municipality,region,roadType,regionArea,timeOfDay,Hirvi,Kuusipeura,Metsäkauris,Metsäpeura,Valkohäntäpeura,Villisika
0,1,2017,12,278773.65900,6.980878e+06,743,14,3,13864807779,68100,1,0,0,0,0,0
1,2,2017,9,397501.00800,7.503084e+06,261,19,4,94206757384,57600,1,0,0,0,0,0
2,3,2017,9,418353.84100,7.514105e+06,261,19,6,94206757384,36000,0,0,0,0,1,0
3,4,2017,12,488970.82700,6.722504e+06,285,8,5,4622479273,68400,1,0,0,0,0,0
4,5,2017,9,418766.58000,7.506931e+06,261,19,6,94206757384,36000,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63936,63937,2021,6,395363.59416,6.687913e+06,92,1,7,9109841425,41400,0,0,1,0,0,0
63937,63938,2021,10,236501.85584,6.691606e+06,445,2,5,10574665809,45000,1,0,0,0,0,0
63938,63939,2021,9,515750.42581,6.946907e+06,778,11,3,17589162923,82308,0,0,1,0,0,0
63939,63940,2021,5,245406.95714,6.712757e+06,202,2,4,10574665809,32400,0,0,1,0,0,0
